# Prepared911 ML Models
## Response Time Prediction, Call Priority Classification, Call Volume Forecasting

This notebook trains 3 ML models and registers them in the Snowflake Model Registry.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
# Per project template requirements, call get_notebook_guide before writing notebook code
from snowflake.ml.utils.notebook_guide import get_notebook_guide
get_notebook_guide()

In [ ]:
SELECT
    ud.RESPONSE_TIME_SEC,
    ud.TRAVEL_TIME_SEC,
    ud.DISTANCE_MILES,
    i.INCIDENT_TYPE_CODE,
    i.SEVERITY,
    i.UNITS_DISPATCHED,
    i.ZONE_ID,
    HOUR(i.DISPATCH_TIMESTAMP) AS HOUR_OF_DAY,
    DAYOFWEEK(i.DISPATCH_TIMESTAMP) AS DAY_OF_WEEK,
    gz.ZONE_TYPE,
    gz.RISK_LEVEL,
    gz.POPULATION
FROM PREPARED911_INTELLIGENCE.RAW.UNIT_DISPATCHES ud
JOIN PREPARED911_INTELLIGENCE.RAW.INCIDENTS i ON ud.INCIDENT_ID = i.INCIDENT_ID
JOIN PREPARED911_INTELLIGENCE.RAW.GEOGRAPHIC_ZONES gz ON i.ZONE_ID = gz.ZONE_ID
WHERE ud.RESPONSE_TIME_SEC IS NOT NULL
    AND ud.RESPONSE_TIME_SEC BETWEEN 60 AND 1800

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

print(f'Dispatch records loaded: {len(df_dispatches)}')
print(f'Columns: {df_dispatches.columns.tolist()}')

## Model 1: Response Time Predictor
GradientBoostingRegressor to predict response time based on incident attributes

In [ ]:
df_rt = df_dispatches.copy()

le_type = LabelEncoder()
le_severity = LabelEncoder()
le_zone = LabelEncoder()
le_zone_type = LabelEncoder()
le_risk = LabelEncoder()

df_rt['TYPE_ENCODED'] = le_type.fit_transform(df_rt['INCIDENT_TYPE_CODE'].fillna('UNKNOWN'))
df_rt['SEVERITY_ENCODED'] = le_severity.fit_transform(df_rt['SEVERITY'].fillna('UNKNOWN'))
df_rt['ZONE_ENCODED'] = le_zone.fit_transform(df_rt['ZONE_ID'].fillna('UNKNOWN'))
df_rt['ZONE_TYPE_ENCODED'] = le_zone_type.fit_transform(df_rt['ZONE_TYPE'].fillna('UNKNOWN'))
df_rt['RISK_ENCODED'] = le_risk.fit_transform(df_rt['RISK_LEVEL'].fillna('UNKNOWN'))

features_rt = ['TYPE_ENCODED', 'SEVERITY_ENCODED', 'ZONE_ENCODED', 'ZONE_TYPE_ENCODED',
               'RISK_ENCODED', 'HOUR_OF_DAY', 'DAY_OF_WEEK', 'UNITS_DISPATCHED', 'POPULATION']

X_rt = df_rt[features_rt].fillna(0)
y_rt = df_rt['RESPONSE_TIME_SEC']

X_train_rt, X_test_rt, y_train_rt, y_test_rt = train_test_split(X_rt, y_rt, test_size=0.2, random_state=42)

model_rt = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
model_rt.fit(X_train_rt, y_train_rt)

y_pred_rt = model_rt.predict(X_test_rt)
mae_rt = mean_absolute_error(y_test_rt, y_pred_rt)
r2_rt = r2_score(y_test_rt, y_pred_rt)

print(f'Response Time Predictor Results:')
print(f'  MAE: {mae_rt:.1f} seconds')
print(f'  R2 Score: {r2_rt:.4f}')

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='PREPARED911_INTELLIGENCE', schema_name='ANALYTICS')

sample_input_rt = X_train_rt.head(10)

mv_rt = registry.log_model(
    model_rt,
    model_name='RESPONSE_TIME_PREDICTOR',
    version_name='V1',
    sample_input_data=sample_input_rt,
    conda_dependencies=['scikit-learn'],
    comment='Predicts emergency response time in seconds based on incident type, zone, severity, time of day'
)

mv_rt.set_metric('mae_seconds', mae_rt)
mv_rt.set_metric('r2_score', r2_rt)
mv_rt.set_metric('training_samples', len(X_train_rt))

print(f'Model RESPONSE_TIME_PREDICTOR V1 registered successfully')

## Model 2: Call Priority Classifier
RandomForestClassifier to classify call priority (1-5) from call attributes

In [ ]:
SELECT
    CALL_ID,
    CALL_TYPE,
    CALL_SUBTYPE,
    PRIORITY,
    LANGUAGE_PRIMARY,
    TRANSLATION_NEEDED,
    CALL_DURATION_SEC,
    CALL_SOURCE,
    ZONE_ID,
    HOUR(CALL_TIMESTAMP) AS HOUR_OF_DAY,
    DAYOFWEEK(CALL_TIMESTAMP) AS DAY_OF_WEEK
FROM PREPARED911_INTELLIGENCE.RAW.INCIDENT_CALLS
WHERE PRIORITY IS NOT NULL

In [ ]:
df_cp = df_calls.copy()

le_call_type = LabelEncoder()
le_subtype = LabelEncoder()
le_source = LabelEncoder()
le_call_zone = LabelEncoder()

df_cp['CALL_TYPE_ENCODED'] = le_call_type.fit_transform(df_cp['CALL_TYPE'].fillna('UNKNOWN'))
df_cp['SUBTYPE_ENCODED'] = le_subtype.fit_transform(df_cp['CALL_SUBTYPE'].fillna('UNKNOWN'))
df_cp['SOURCE_ENCODED'] = le_source.fit_transform(df_cp['CALL_SOURCE'].fillna('UNKNOWN'))
df_cp['ZONE_ENCODED'] = le_call_zone.fit_transform(df_cp['ZONE_ID'].fillna('UNKNOWN'))
df_cp['TRANSLATION_INT'] = df_cp['TRANSLATION_NEEDED'].astype(int)

features_cp = ['CALL_TYPE_ENCODED', 'SUBTYPE_ENCODED', 'SOURCE_ENCODED',
               'ZONE_ENCODED', 'HOUR_OF_DAY', 'DAY_OF_WEEK', 'TRANSLATION_INT']

X_cp = df_cp[features_cp].fillna(0)
y_cp = df_cp['PRIORITY']

X_train_cp, X_test_cp, y_train_cp, y_test_cp = train_test_split(X_cp, y_cp, test_size=0.2, random_state=42)

model_cp = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
model_cp.fit(X_train_cp, y_train_cp)

y_pred_cp = model_cp.predict(X_test_cp)
accuracy_cp = accuracy_score(y_test_cp, y_pred_cp)

print(f'Call Priority Classifier Results:')
print(f'  Accuracy: {accuracy_cp:.4f}')

In [ ]:
sample_input_cp = X_train_cp.head(10)

mv_cp = registry.log_model(
    model_cp,
    model_name='CALL_PRIORITY_CLASSIFIER',
    version_name='V1',
    sample_input_data=sample_input_cp,
    conda_dependencies=['scikit-learn'],
    comment='Classifies 911 call priority (1-5) based on call type, subtype, source, zone, and time factors'
)

mv_cp.set_metric('accuracy', accuracy_cp)
mv_cp.set_metric('training_samples', len(X_train_cp))

print(f'Model CALL_PRIORITY_CLASSIFIER V1 registered successfully')

## Model 3: Call Volume Forecaster
GradientBoostingRegressor to forecast hourly call volumes

In [ ]:
SELECT
    DATE_TRUNC('hour', CALL_TIMESTAMP) AS CALL_HOUR,
    HOUR(CALL_TIMESTAMP) AS HOUR_OF_DAY,
    DAYOFWEEK(CALL_TIMESTAMP) AS DAY_OF_WEEK,
    MONTH(CALL_TIMESTAMP) AS MONTH_NUM,
    COUNT(*) AS CALL_COUNT
FROM PREPARED911_INTELLIGENCE.RAW.INCIDENT_CALLS
GROUP BY 1, 2, 3, 4
ORDER BY CALL_HOUR

In [ ]:
df_vol = df_volume.copy()

df_vol['IS_WEEKEND'] = (df_vol['DAY_OF_WEEK'].isin([6, 7])).astype(int)
df_vol['IS_NIGHT'] = ((df_vol['HOUR_OF_DAY'] >= 22) | (df_vol['HOUR_OF_DAY'] <= 5)).astype(int)
df_vol['HOUR_SIN'] = np.sin(2 * np.pi * df_vol['HOUR_OF_DAY'] / 24)
df_vol['HOUR_COS'] = np.cos(2 * np.pi * df_vol['HOUR_OF_DAY'] / 24)
df_vol['DOW_SIN'] = np.sin(2 * np.pi * df_vol['DAY_OF_WEEK'] / 7)
df_vol['DOW_COS'] = np.cos(2 * np.pi * df_vol['DAY_OF_WEEK'] / 7)

features_vol = ['HOUR_OF_DAY', 'DAY_OF_WEEK', 'MONTH_NUM', 'IS_WEEKEND',
                'IS_NIGHT', 'HOUR_SIN', 'HOUR_COS', 'DOW_SIN', 'DOW_COS']

X_vol = df_vol[features_vol].fillna(0)
y_vol = df_vol['CALL_COUNT']

X_train_vol, X_test_vol, y_train_vol, y_test_vol = train_test_split(X_vol, y_vol, test_size=0.2, random_state=42)

model_vol = GradientBoostingRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model_vol.fit(X_train_vol, y_train_vol)

y_pred_vol = model_vol.predict(X_test_vol)
mae_vol = mean_absolute_error(y_test_vol, y_pred_vol)
r2_vol = r2_score(y_test_vol, y_pred_vol)

print(f'Call Volume Forecaster Results:')
print(f'  MAE: {mae_vol:.2f} calls')
print(f'  R2 Score: {r2_vol:.4f}')

In [ ]:
sample_input_vol = X_train_vol.head(10)

mv_vol = registry.log_model(
    model_vol,
    model_name='CALL_VOLUME_FORECASTER',
    version_name='V1',
    sample_input_data=sample_input_vol,
    conda_dependencies=['scikit-learn'],
    comment='Forecasts hourly 911 call volumes based on time-of-day, day-of-week, and seasonal patterns'
)

mv_vol.set_metric('mae_calls', mae_vol)
mv_vol.set_metric('r2_score', r2_vol)
mv_vol.set_metric('training_samples', len(X_train_vol))

print(f'Model CALL_VOLUME_FORECASTER V1 registered successfully')

In [ ]:
print('\n' + '='*60)
print('ALL MODELS REGISTERED SUCCESSFULLY')
print('='*60)
print(f'\n1. RESPONSE_TIME_PREDICTOR V1 (MAE: {mae_rt:.1f}s, R2: {r2_rt:.4f})')
print(f'2. CALL_PRIORITY_CLASSIFIER V1 (Accuracy: {accuracy_cp:.4f})')
print(f'3. CALL_VOLUME_FORECASTER V1 (MAE: {mae_vol:.2f} calls, R2: {r2_vol:.4f})')
print('\nModels are available in: PREPARED911_INTELLIGENCE.ANALYTICS')